# 03 Simulation
Runs single and double SRG removal scenarios and computes network metrics for each.

In [5]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.network.build import prepare_edge_df
from src.simulation.failures import (
    compute_metrics_for_graph,
    remove_single_srgs,
    remove_double_srgs,
)

In [6]:
# Load data and build graphs
# file paths
import yaml
import pickle

with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

# Reimport directories and load
NODE_DIR = config["paths"]["node_list"]
EDGE_DIR = config["paths"]["edge_list"]
IXP_DIR = config["paths"]["ixp_list"]
SRG_DIR = config["paths"]["srg_list"]
SITE_DIR = config["paths"]["site_list"]
GRAPH_DIR = config["paths"]["graph_folder"]
TABULAR_DIR = config["paths"]["tabular_folder"]

node_df = pd.read_csv(NODE_DIR)
edge_df = pd.read_csv(EDGE_DIR)
edge_df = prepare_edge_df(edge_df)
srg_city_df = pd.read_csv(SRG_DIR)
ixp_df = pd.read_csv(IXP_DIR)
site_df = pd.read_csv(SITE_DIR, comment='#')

# list of sites
site_list = list(site_df['site_id'])

# processed srg df
srg_c_df = pd.read_csv(f'{TABULAR_DIR}/srg_c.csv')

# load graph objects
with open(f"{GRAPH_DIR}/G_full.pickle", "rb") as f:
    G_full = pickle.load(f)

print('Graph ready. Running simulations...')

Graph ready. Running simulations...


In [7]:
# Baseline metrics
baseline = compute_metrics_for_graph(G_full, site_list)
baseline['scenario'] = 'baseline'

print('Baseline components:', baseline['n_components'])
print('Baseline largest CC frac: ', baseline['largest_cc_frac'])
print('Baseline diameter:', baseline['diameter'])
print('Baseline average shortest path:', baseline['avg_shortest_path'])
print('Baseline average clustering coefficient:', baseline['avg_cluster_coeff'])

Baseline components: 1
Baseline largest CC frac:  1.0
Baseline diameter: 8
Baseline average shortest path: 3.58887381275441
Baseline average clustering coefficient: 0.27562189054726366


In [8]:
# Single SRLG removal (conservative only)
print('Running single SRG removal (conservative)...')
single_df = remove_single_srgs(srg_c_df, G_full, baseline, site_list)
print('Done.', len(single_df) -1, 'SRLG removals')
single_df.head()

Running single SRG removal (conservative)...
Done. 51 SRLG removals


,nodes_total,edges_total,n_components,largest_cc_frac,in_lcc,present_sites,diameter,avg_shortest_path,avg_cluster_coeff,aarnet_nodes,...,vocus_n_components,vocus_largest_cc_frac,vocus_in_lcc,vocus_present_sites,vocus_diameter,vocus_avg_shortest_path,vocus_avg_cluster_coeff,scenario,srg_id,edges_removed
0,67,171,1,1.0,13,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",8,3.588874,0.275622,11,...,1,1.0,11,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",4.0,2.090909,0.518182,baseline,NaN,NaN
1,67,165,1,1.0,13,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",8,3.724559,0.244776,11,...,1,1.0,11,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",4.0,2.145455,0.533333,single removal,ADL_DRW_TER_1,6.0
2,67,167,1,1.0,13,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",8,3.588874,0.275622,11,...,1,1.0,11,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",4.0,2.090909,0.518182,single removal,ADL_SMAP_SUB_1,4.0
3,67,167,1,1.0,13,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",8,3.651741,0.223383,11,...,1,1.0,11,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",5.0,2.181818,0.366667,single removal,BNE_ROK_TER_1,4.0
4,67,170,1,1.0,13,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",8,3.588874,0.275622,11,...,1,1.0,11,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",4.0,2.090909,0.518182,single removal,BNE_ROK_TER_2,1.0


In [9]:
# Double SRLG removal (conservative only)
# NOTE: O(n^2) — can take several minutes for large SRG sets
print(f'Running double SRG removal (conservative): {len(srg_c_df)} SRLGs,',
      f'{len(srg_c_df) * (len(srg_c_df) - 1) // 2} pairs...')
double_df = remove_double_srgs(srg_c_df, G_full, baseline, site_list)
print('Done.', len(double_df) -1, 'pairs')
double_df.head()

Running double SRG removal (conservative): 51 SRLGs, 1275 pairs...
Done. 1275 pairs


,nodes_total,edges_total,n_components,largest_cc_frac,in_lcc,present_sites,diameter,avg_shortest_path,avg_cluster_coeff,aarnet_nodes,...,vocus_present_sites,vocus_diameter,vocus_avg_shortest_path,vocus_avg_cluster_coeff,scenario,srg_id_1,srg_id_2,edges_removed,edges_removed_1,edges_removed_2
0,67,160,4,0.895522,12,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",inf,inf,0.250746,11,...,"[ADL_1, BNE_1, CBR_1, DRW_1, MEL_1, PER_1, SYD...",inf,inf,0.530303,double removal,HBA_LST_TER_1,HBA_LST_TER_2,11.0,5.0,6.0
1,67,160,3,0.910448,13,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",inf,inf,0.250746,11,...,"[ADL_1, BNE_1, CBR_1, DRW_1, MEL_1, PER_1, SYD...",inf,inf,0.530303,double removal,HBA_LST_TER_1,MEL_LST_SUB_2,11.0,5.0,6.0
2,67,161,2,0.925373,13,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",inf,inf,0.250746,11,...,"[ADL_1, BNE_1, CBR_1, DRW_1, MEL_1, PER_1, SYD...",inf,inf,0.530303,double removal,HBA_LST_TER_2,MEL_LST_SUB_2,12.0,6.0,6.0
3,67,161,3,0.940299,12,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",inf,inf,0.252736,11,...,"[ADL_1, BNE_1, CBR_1, DRW_1, MEL_1, PER_1, SYD...",inf,inf,0.530303,double removal,MEL_LST_SUB_1,MEL_LST_SUB_2,10.0,4.0,6.0
4,67,153,4,0.955224,13,"[ADL_1, BNE_1, CBR_1, DRW_1, HBA_1, MEL_1, PER...",inf,inf,0.200498,11,...,"[ADL_1, BNE_1, DRW_1, HBA_1, MEL_1, PER_1, SYD...",inf,inf,0.393939,double removal,MEL_CBR_TER_2,SYD_CBR_TER_2,18.0,9.0,9.0


In [10]:
# Save results
single_df.to_csv(f'{TABULAR_DIR}/single.csv', index=False)
double_df.to_csv(f'{TABULAR_DIR}/double.csv', index=False)
print('Saved.')

Saved.
